**What this notebook does:**
1. Cleans up the datasets for data visualization.


2. Creates a 'competition score', so you can compare a company to its same cateogries within the same geographical area.
3. Creates an 'overall z-score of category', so you can compare the company to its category across the state
4. Creats a 'city relative score' that is a z-score of the average rating in compairison to the area in which the company is located.
3. Creates a '95% confidence interval' for the probable average ratings a company could have.
4. Creates 2 datasets that can be used with these new columns of different sizes.

In [1]:
#TODO 3, 6 and code review

**Step 1:** clean datasets before merging and feature engineering, so Tableau is displayed without outliers and NULL data.

In [2]:
import numpy as np
from sklearn.neighbors import BallTree
import pandas as pd
import numpy as np
import gzip
import json

pd.set_option('display.max_columns', None)

def parse(path):
  g = open('Data/' + path, 'r')
  for l in g:
    yield json.loads(l)

def parse_first_n(path, n=10000):
    g = open('Data/' + path, 'r')
    for i, l in enumerate(g):
        if i >= n:
            break
        yield json.loads(l)

In [3]:
texas_metadata = pd.DataFrame(parse("meta-Texas.json"))
texas_reviews = pd.DataFrame(parse_first_n("review-Texas.json", n=1000000)) # is not cleaned yet
#add more or less reviews depending on how well tableu responds to the data.


**Clean the data**

In [4]:
texas_reviews = texas_reviews.dropna(subset=['rating']) # Drop rows where 'rating' is NaN

In [5]:
texas_metadata = texas_metadata.dropna(subset=['gmap_id', 'name']).drop_duplicates(subset=['gmap_id'], keep='first')



In [6]:
texas_metadata = texas_metadata[texas_metadata['state'] != 'Permanently closed']


In [7]:
texas_metadata = texas_metadata[(texas_metadata['latitude'] >= 25.5) & (texas_metadata['latitude'] <= 36.40)
                            & (texas_metadata['longitude'] >= -106.50) & (texas_metadata['longitude'] <= -93.31)]


In [8]:
texas_metadata.shape[0]

416224

**Add a city column to each company**

In [9]:
def standardize_rating_by_city(df):
    """
    Standardizes average ratings relative to each city's mean and standard deviation.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame with 'avg_rating' and 'city' columns
        
    Returns:
    --------
    pandas.Series
        Z-scores (standardized ratings) for each row
    """
    # Calculate city-level statistics
    city_stats = df.groupby('city')['avg_rating'].agg(['mean', 'std'])
    
    # Map city statistics to each row
    df_with_stats = df.copy()
    df_with_stats['city_mean'] = df['city'].map(city_stats['mean'])
    df_with_stats['city_std'] = df['city'].map(city_stats['std']) #map is like a mini merge
    
    # Calculate z-score: (value - mean) / std
    # Handle cases where std is 0 (all ratings in city are the same)
    standardized = (df_with_stats['avg_rating'] - df_with_stats['city_mean']) / df_with_stats['city_std'].replace(0, 1)
    
    return standardized

# Apply the function to add standardized ratings


In [10]:
path = 'Data/simplemaps_uscities_basicv1.93/uscities.csv'

us_cities = pd.read_csv(path)



# Convert to radians


category_df = texas_metadata.copy()
#THIS WILL FIRST ADD A CITY COLUMN TO OUR METADATA SO WE CAN GROUP
cities_rad = np.radians(us_cities[['lat', 'lng']])
companies_rad = np.radians(category_df[['latitude', 'longitude']])

# Build BallTree
tree = BallTree(cities_rad, metric='haversine')

# Find nearest city for each company
distances, indices = tree.query(companies_rad, k=1)

# Assign city name
nearest_city_names = us_cities.iloc[indices.flatten()]['city'].reset_index(drop=True)

category_df = category_df.reset_index(drop=True)
category_df['city'] = nearest_city_names
#now has a city column on category_df which we can use groupby to attatch a new column for
category_df['demand'] = category_df.groupby('city')['gmap_id'].transform('count')
df_for_later = category_df.copy()
category_df['standardized_city_rating'] = standardize_rating_by_city(category_df)
#make sure this column gets included in the final output
#Extract all unique category labels from california_metadata
category_df = category_df.explode('category')
#Convert everything to lower case and remove blank space 
category_df['category'] = category_df['category'].str.lower().str.strip()






**#TODO work with Mia to concatinate categories together in this cell's position**

In [11]:
category_stats = category_df.groupby('category')['avg_rating'].agg(['mean', 'std'])
#rn category stats is just grouped by category as a whole
# Map city statistics to each row
df_with_stats = category_df.copy()

df_with_stats['cat_mean'] = df_with_stats['category'].map(category_stats['mean'])
df_with_stats['cat_std'] = df_with_stats['category'].map(category_stats['std'])

# Calculate z-score: (value - mean) / std
# Handle cases where std is 0 (all ratings in city are the same)
category_df['Entire_category_score'] = (df_with_stats['avg_rating'] - df_with_stats['cat_mean']) / df_with_stats['cat_std'].replace(0, 1)
#this makes that 'Entire_category_score'


In [12]:
# # # #Remove tiny categories 
category_counts= category_df.get('category').value_counts()


# Setting threshold for the category 
category_counts.describe()
category_counts_threshold = 35 #median, we could alter this if there are too many categories for our deomo


# Map category_count to each category in the df
category_df['category_count'] = category_df['category'].map(category_counts)
category_df = category_df[category_df.get('category_count') >= category_counts_threshold]
category_df = category_df.sort_values(by='category_count', ascending=False)


**Aggregate to each category within each city**

In [13]:

# Calculate mean, std, and count for each category within each city
category_stats = category_df.groupby(["city", "category"])['avg_rating'].agg(['mean', 'std', 'count']).reset_index()
category_stats.columns = ['city', 'category', 'cc_mean', 'cc_std', 'cc_count']  # rename columns for clarity
#use these for our average rating of category in the city column

In [14]:

# Merge back into category data set (here category df is just the exploded metadata with city and category count)
mcategory_df = category_df.merge(category_stats, on=['city', 'category'], how='inner') #merge on both city and category
mcategory_df.shape[0]
mcategory_df['cc_count'] = (
    mcategory_df.groupby(['city', 'category'])['gmap_id']
      .transform('nunique')
)
mcategory_df
mcategory_df.drop(columns=[ 'category_count'])


,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url,city,demand,standardized_city_rating,Entire_category_score,cc_mean,cc_std,cc_count
0,Wendy's,"Wendy's, 4544 Bissonnet St, Bellaire, TX 77401",0x8640c1a3aa5a99fb:0xc5c2a6a449088ff,Fast-food burger chain serving sides such as c...,29.715020,-95.453964,restaurant,4.2,538,$,"[[Friday, 6:30AM–11PM], [Saturday, 6:30AM–11PM...","{'Service options': ['Delivery', 'Drive-throug...",Open ⋅ Closes 11PM,"[0x8640c18954206d2f:0x4e15587d93e4318, 0x8640c...",https://www.google.com/maps/place//data=!4m2!3...,Bellaire,3692,-0.021549,0.153994,4.124352,0.441680,193
1,McDonald's,"McDonald's, 612 Tennessee Ave, Dalhart, TX 79022",0x8705b6c7471ba2f9:0x847da2fa99d7536c,"Classic, long-running fast-food chain known fo...",36.060724,-102.514050,restaurant,3.5,888,$,"[[Sunday, 5AM–11PM], [Monday, 5AM–11PM], [Tues...","{'Service options': ['Curbside pickup', 'Drive...",Closed ⋅ Opens 5AM Mon,"[0x8705b6c6362a4f81:0x369aaaa1d8a4be3, 0x8705b...",https://www.google.com/maps/place//data=!4m2!3...,Dalhart,150,-1.108889,-1.302278,4.144444,0.485054,9
2,George's Cafe,"George's Cafe, 2337 Gravel Dr, Fort Worth, TX ...",0x864e797751d6b4e3:0x97ad8d21c39c7c47,None,32.792568,-97.217853,restaurant,4.4,57,$,"[[Sunday, Closed], [Monday, 7AM–3PM], [Tuesday...","{'Service options': ['Takeout', 'Dine-in'], 'H...",Closed ⋅ Opens 7AM Mon,"[0x864e79827fecf4cd:0x6d4e478c5f9b6bdf, 0x864e...",https://www.google.com/maps/place//data=!4m2!3...,Richland Hills,1024,0.313438,0.570071,4.058140,0.573723,43
3,Shell,"Shell, 1700 N, TX-121, Grapevine, TX 76051",0x864c2b90ea42211f:0x1173d80852dfb390,None,32.955463,-97.037509,restaurant,2.6,48,$$,"[[Sunday, Open 24 hours], [Monday, Open 24 hou...",{'Accessibility': ['Wheelchair accessible entr...,Open 24 hours,"[0x864c2b178010cfaf:0xe1ca8b0ce7b475cc, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,Grapevine,1500,-2.372909,-3.174627,4.050400,0.569752,125
4,Taco Bell,"Taco Bell, 4151 Lyndon B Johnson Fwy, Dallas, ...",0x864c272e007a2663:0x2e8ddfcf246d6080,Fast-food chain serving Mexican-inspired fare ...,32.925694,-96.840475,restaurant,3.8,525,$,"[[Sunday, 9PM–1AM], [Monday, 9AM–1AM], [Tuesda...","{'Service options': ['Delivery', 'Drive-throug...",Open ⋅ Closes 1AM,"[0x864c20e717219c17:0x355db863db0d9003, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,Farmers Branch,2564,-0.536650,-0.678161,4.028829,0.519858,111
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
952617,Kingwood Photo Lab,"Kingwood Photo Lab, 2714 W Lake Houston Pkwy S...",0x8640acd7767ac76b:0xeda7828707704177,None,30.046727,-95.181765,photo restoration service,4.6,32,None,"[[Tuesday, 10AM–5PM], [Wednesday, 10AM–5PM], [...","{'Service options': ['Onsite services', 'Onlin...",Closed ⋅ Opens 10AM,"[0x8640c69f8ee808b7:0x8f139fcae0ee07db, 0x8640...",https://www.google.com/maps/place//data=!4m2!3...,Atascocita,1367,0.481110,-0.238240,4.600000,NaN,1
952618,Texas Finishing Co,"Texas Finishing Co, 1801 Surveyor Blvd, Carrol...",0x864c26b846232d83:0xffcaa14639ebaa6e,None,32.960964,-96.847399,metal finisher,4.1,8,None,"[[Tuesday, 7AM–4PM], [Wednesday, 7AM–4PM], [Th...",{'Accessibility': ['Wheelchair accessible entr...,Opens soon ⋅ 7AM,"[0x864e9067c132aeb3:0xe1bf7a51153d24c3, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,Addison,3779,-0.201008,-0.122290,4.100000,NaN,1
952619,Frets & Keys,"Frets & Keys, 380 Litchfield Ln, Houston, TX 7...",0x8640c35809834487:0x21fa0e8a2f6532a2,None,29.765187,-95.550281,piano tuning service,5.0,1,None,None,None,None,"[0x8640de5ba87dd071:0x72325c60707f4cb1, 0x8640...",https://www.google.com/maps/place//data=!4m2!3...,Bunker Hill Village,2916,1.092466,0.679453,5.000000,NaN,1
952620,Guaranty Bank & Trust,"Guaranty Bank & Trust, 655 W Grand Pkwy S, Kat...",0x86412743c8c39683:0xf106f001466ec5d5,None,29.771322,-95.776062,currency exchange service,4.9,8,N

In [15]:
mcategory_df.columns #lets use cc_mean and keep it as a column 
#so we can do 5 - df['cc_mean'] to get our quality penalty
#TODO check if this includes the 'demand' column and the cc_mean

Index(['name', 'address', 'gmap_id', 'description', 'latitude', 'longitude',
       'category', 'avg_rating', 'num_of_reviews', 'price', 'hours', 'MISC',
       'state', 'relative_results', 'url', 'city', 'demand',
       'standardized_city_rating', 'Entire_category_score', 'category_count',
       'cc_mean', 'cc_std', 'cc_count'],
      dtype='object')

In [16]:

# competition score = normalized rating in local area relative to competition
# If less than 2 companies in the city-category combo, mark as insufficient data
mcategory_df['competition_score'] = mcategory_df.apply(
    lambda row: 
        (row['avg_rating'] - row['cc_mean']) / row['cc_std']
        if (row['cc_count'] >= 2 and row['cc_std'] != 0)
        else np.nan,
    axis=1
)

mcategory_df = mcategory_df.groupby('gmap_id').agg({
    'competition_score': 'mean',
    'standardized_city_rating': 'mean',
    'Entire_category_score': 'mean',
    'cc_mean': 'mean',
    'cc_count': 'mean',
    'demand': 'mean'
    
    #this gives us cc_mean as a col to use 'Targetability'
}).reset_index()

mcategory_df #The reason that the number dropped is becuase explode will create multiple rows and grouping by gmap_id will reduce that back


,gmap_id,competition_score,standardized_city_rating,Entire_category_score,cc_mean,cc_count,demand
0,0x0:0x3de3e25d19fc7da6,-0.142294,-0.708316,-0.639810,3.764031,8.714286,1589.0
1,0x0:0x48c466d0f3de5f5a,0.468495,0.879989,0.514483,4.775697,31.200000,3164.0
2,0x0:0x554de8957fed30a1,0.116645,-0.108407,0.196436,4.077594,49.400000,1819.0
3,0x0:0x5702eb6d2d1301db,1.333021,1.353825,1.148322,4.131818,22.000000,916.0
4,0x1000000000000001:0xe85d5b0e25befcb7,1.292591,0.552496,0.460215,3.850000,4.000000,1362.0
...,...,...,...,...,...,...,...
409730,0xaceec0ea6c38bcfd:0xf291bc79aa31fa60,0.707107,1.233789,0.692342,4.350000,2.000000,212.0
409731,0xad33c203e03d89f9:0xb8ee8c778a581ca4,0.519618,1.140207,0.707797,4.839474,10.500000,1492.0
409732,0xd0c7e1a240869ad:0xafa498adc8911c71,0.712855,1.011768,0.736033,4.425439,106.000000,7526.0
409733,0xdce6e3016b65401:0xf220f369359026f2,1.421060,1.102625,1.339574,4.195769,35.000000,2889.0


In [17]:
#We will merge all of this back into texas metadata with an inner merge to add the new z-score and CI columns for the Tableu dataset.

company_stats = (
    texas_reviews #first time using texas reviews, we should definetly do a left merge for CI, since limmited data n=1000000
        .groupby('gmap_id')['rating']
        .agg(['mean', 'std', 'count'])
        .reset_index()
)

def compute_ci(row):
    mean = row['mean']
    std = row['std']
    n = row['count']
    
    if n == 0:
        return None
    
    margin = 1.96 * (std / np.sqrt(n))
    return (mean - margin, mean + margin)

company_stats['CI'] = company_stats.apply(compute_ci, axis=1)

mtexas_metadata = texas_metadata.merge(
    company_stats[['gmap_id', 'CI']],
    on='gmap_id',
    how='inner'
)

# mcategory_df['CI']= mcategory_df['gmap_id'].apply(individual_company_rating_confidence_interval)
# mcategory_df #THIS TAKES TO LONG SPEED IT UP AND THEN MERGE BACK TO TEXAS METADATA



The issue is that out output code has a list of categories in the category column, we don't want only one category using the .explode method becasue that would duplicate out gmap_ids. This would cause our data visualization to crash.

**<u>The list format of categories doesen't matter becuse the competition score already finds the competition relative to all of the companies categories, This score is the most important as it is honestly more acurate at it takes into account multiple varibales rather than just one, calculates them separately and then avereges the scores.**



In [26]:


final_texas_metadata = texas_metadata.merge(mtexas_metadata[['gmap_id', 'CI']], on='gmap_id', how='left').merge(mcategory_df[['gmap_id', 'competition_score', 'standardized_city_rating', 'Entire_category_score', 'cc_count', 'demand', 'cc_mean']], on='gmap_id', how='left')
final_texas_metadata['competition_score'] = final_texas_metadata['competition_score'].fillna("not enough data")
final_texas_metadata
final_texas_metadata = final_texas_metadata.merge(df_for_later[['gmap_id','city']], on= 'gmap_id') #WTF IS WRONG
#shuold be > 409k
final_texas_metadata['category'] = final_texas_metadata['category'].apply(lambda x: tuple(x) if x is not None else None)

****BELOW ARE 2 FINAL DATASETS. ONE WITH ONLY THE COMPANIES IN TEXAS REVIEWS WITH NO NAN VALUES this is small_... AND THE OTHER INCLUDES ALL THE DATA IN META BUT NAN VALUES. this is big_.... Choose which one to use BASED ON HOW WELL TABLEAU HANDLES A LOT OF DATA.****

In [27]:
small_test_final_texas_metadata = final_texas_metadata.dropna(subset=['CI'])
small_test_final_texas_metadata = small_test_final_texas_metadata[~small_test_final_texas_metadata['CI'].apply(lambda x: any(pd.isna(i) for i in x))]
small_test_final_texas_metadata['CI'] = small_test_final_texas_metadata['CI'].apply(lambda x: (x[0],5.0) if x[1] > 5.0 else (1.0, x[1]) if x[0] < 1.0 else x)
small_test_final_texas_metadata = small_test_final_texas_metadata.rename(columns={
    'CI': 'CI_for_avg_rating',
    'Entire_category_score': 'category_relative_score'
})

# Merge in city and category from category_df (which has the city/category mapping)






**Adjusting confidence intervals**

In [28]:
big_test_final_texas_metadata = final_texas_metadata.copy()

def clamp_ci(x): #this fucniton is only needed for big due to NaN handling
    # Skip NaN or non-tuples
    if not isinstance(x, (tuple, list)):
        return x
    
    lower, upper = x
    
    # Clamp values
    lower = max(1.0, lower)
    upper = min(5.0, upper)
    
    return (lower, upper)

big_test_final_texas_metadata['CI'] = (
    big_test_final_texas_metadata['CI'].apply(clamp_ci)
)

# Rename columns
big_test_final_texas_metadata = big_test_final_texas_metadata.rename(columns={
    'CI': 'CI_for_avg_rating',
    'Entire_category_score': 'category_relative_score'
})


In [29]:
print(big_test_final_texas_metadata.shape[0], small_test_final_texas_metadata.shape[0]) #big vs small difference

416224 34376


In [30]:
# Calculate opportunity_score per city-category combination
def calculate_opportunity_score(df):
    """
    Calculate opportunity score per city-category combination.
    
    Formula: ((5 - cc_mean) * demand) / cc_count
    """
    # Use the category column (handle case where there might be duplicates)

    
    opportunity_scores = df.groupby(['city', 'category']).apply( #category isnt hashable 
        lambda group: ((5 - group['cc_mean'].iloc[0]) * group['demand'].iloc[0]) / group['cc_count'].iloc[0],
        include_groups=False
    ).reset_index(name='opportunity_score')
    opportunity_scores.columns = ['city', 'category', 'opportunity_score']

    # Merge back to preserve all rows
    result = df.merge(
        opportunity_scores,
        on=['city', 'category'],
        how='left'
    )
    return result

small_test_final_texas_metadata = calculate_opportunity_score(small_test_final_texas_metadata)
big_test_final_texas_metadata = calculate_opportunity_score(big_test_final_texas_metadata)

In [32]:
small_test_final_texas_metadata.isna().sum()

name                            0
address                      1366
gmap_id                         0
description                 32674
latitude                        0
longitude                       0
category                      178
avg_rating                      0
num_of_reviews                  0
price                       32723
hours                        8532
MISC                         8109
state                        8053
relative_results             4408
url                             0
CI_for_avg_rating               0
competition_score               0
standardized_city_rating      729
category_relative_score       719
cc_count                      719
demand                        719
cc_mean                       719
city                            0
opportunity_score             719
dtype: int64

**try and add this funciton above**

In [33]:
big_test_final_texas_metadata.isna().sum()
#city is not null at all, so we are good to group by it

name                             0
address                       8551
gmap_id                          0
description                 340106
latitude                         0
longitude                        0
category                      1574
avg_rating                       0
num_of_reviews                   0
price                       340669
hours                        85151
MISC                         67655
state                       117313
relative_results             37575
url                              0
CI_for_avg_rating           379338
competition_score                0
standardized_city_rating      6551
category_relative_score       6489
cc_count                      6489
demand                        6489
cc_mean                       6489
city                             0
opportunity_score             6489
dtype: int64

In [ ]:
small_test_final_texas_metadata
check = (
    small_test_final_texas_metadata.groupby(['city', 'category'])
      .agg({
          'cc_mean': 'nunique',
          'demand': 'nunique',
          'cc_count': 'nunique'
      })
)

check.head()

problem_groups = check[
    (check['cc_mean'] > 1) |
    (check['demand'] > 1) |
    (check['cc_count'] > 1)
]

problem_groups
#No duplicates AGTG

,,cc_mean,demand,cc_count
city,category,,,


In [41]:

check2 = (
    big_test_final_texas_metadata.groupby(['city', 'category'])
      .agg({
          'cc_mean': 'nunique',
          'demand': 'nunique',
          'cc_count': 'nunique'
      })
)

check2.head()

problem_groups = check2[
    (check2['cc_mean'] > 1) |
    (check2['demand'] > 1) |
    (check2['cc_count'] > 1)
]

problem_groups
#legit one so this is trivial

,,cc_mean,demand,cc_count
city,category,,,
Denton,"(Coffee shop, Cafe, Coffee store, Espresso bar)",2,1,1


***FINAL DATA FRAMES!!!!!***

In [43]:
big_test_final_texas_metadata.head()

,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url,CI_for_avg_rating,competition_score,standardized_city_rating,category_relative_score,cc_count,demand,cc_mean,city,opportunity_score
0,Timewise Food Store,"Timewise Food Store, 1600 W Church St, Livings...",0x8638869e6b4e3529:0xe8d257447fe41672,None,30.713368,-94.954344,"(Convenience store,)",4.8,4,None,"[[Thursday, Open 24 hours], [Friday, Open 24 h...","{'Service options': ['In-store shopping', 'Del...",Open 24 hours,"[0x863886bab3f9bb05:0x28a8062d0597dd34, 0x8638...",https://www.google.com/maps/place//data=!4m2!3...,"(4.429219701353091, 5.0)",1.696803,0.972932,1.587334,14.00,364.0,3.850000,Livingston,29.900000
1,Goldstar Transit,"Goldstar Transit, 4415 W Dickson Ln, Little El...",0x864c3748dcc1c12d:0xbf904a61f262cf9b,None,33.159363,-96.975571,"(Transportation service,)",4.5,4,None,"[[Thursday, 6AM–6PM], [Friday, 6AM–6PM], [Satu...",{'Accessibility': ['Wheelchair accessible entr...,Open ⋅ Closes 6PM,"[0x864c374855555555:0x3abb669a098bb235, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,"(3.858439402706183, 5.0)",not enough data,0.243142,0.702763,1.00,14.0,4.500000,Lakewood Village,7.000000
2,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,"(Pharmacy, Drug store, Medical supply store, V...",3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,"(2.773104882616715, 3.810228450716618)",0.252294,-1.271692,-0.716826,2.25,161.0,3.262500,Little Elm,124.327778
3,Luminous Logistics,"Luminous Logistics, 3838 W Miller Rd, Garland,...",0x864ea0993bffffff:0xb50b5bb2fccf9d9b,None,32.893678,-96.688611,"(Delivery service,)",2.3,8,None,None,None,None,"[0x864ea09938bb619f:0x1b6902de2a2f3f96, 0x864e...",https://www.google.com/maps/place//data=!4m2!3...,"(1.382640020906352, 3.117359979093648)",not enough data,-2.768392,-3.818610,1.00,3040.0,2.300000,Garland,8208.000000
4,Pacesetter Personnel Services,"Pacesetter Personnel Services, 2300 Valley Vie...",0x864e819d99a1ff99:0xeee31cc82854286c,None,32.839795,-97.020987,"(Employment agency,)",2.1,7,None,None,None,None,"[0x864e9d6ea0c9089f:0x6f90f8b0b092af49, 0x864e...",https://www.google.com/maps/place//data=!4m2!3...,"(1.16055216123327, 3.1251621244810153)",-1.696591,-2.680228,-2.006615,28.00,3590.0,3.642857,Irving,174.005102


In [44]:
small_test_final_texas_metadata.head()

,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url,CI_for_avg_rating,competition_score,standardized_city_rating,category_relative_score,cc_count,demand,cc_mean,city,opportunity_score
0,Timewise Food Store,"Timewise Food Store, 1600 W Church St, Livings...",0x8638869e6b4e3529:0xe8d257447fe41672,None,30.713368,-94.954344,"(Convenience store,)",4.8,4,None,"[[Thursday, Open 24 hours], [Friday, Open 24 h...","{'Service options': ['In-store shopping', 'Del...",Open 24 hours,"[0x863886bab3f9bb05:0x28a8062d0597dd34, 0x8638...",https://www.google.com/maps/place//data=!4m2!3...,"(4.429219701353091, 5.0)",1.696803,0.972932,1.587334,14.00,364.0,3.850000,Livingston,29.900000
1,Goldstar Transit,"Goldstar Transit, 4415 W Dickson Ln, Little El...",0x864c3748dcc1c12d:0xbf904a61f262cf9b,None,33.159363,-96.975571,"(Transportation service,)",4.5,4,None,"[[Thursday, 6AM–6PM], [Friday, 6AM–6PM], [Satu...",{'Accessibility': ['Wheelchair accessible entr...,Open ⋅ Closes 6PM,"[0x864c374855555555:0x3abb669a098bb235, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,"(3.858439402706183, 5.0)",not enough data,0.243142,0.702763,1.00,14.0,4.500000,Lakewood Village,7.000000
2,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,"(Pharmacy, Drug store, Medical supply store, V...",3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,"(2.773104882616715, 3.810228450716618)",0.252294,-1.271692,-0.716826,2.25,161.0,3.262500,Little Elm,124.327778
3,Luminous Logistics,"Luminous Logistics, 3838 W Miller Rd, Garland,...",0x864ea0993bffffff:0xb50b5bb2fccf9d9b,None,32.893678,-96.688611,"(Delivery service,)",2.3,8,None,None,None,None,"[0x864ea09938bb619f:0x1b6902de2a2f3f96, 0x864e...",https://www.google.com/maps/place//data=!4m2!3...,"(1.382640020906352, 3.117359979093648)",not enough data,-2.768392,-3.818610,1.00,3040.0,2.300000,Garland,8208.000000
4,Pacesetter Personnel Services,"Pacesetter Personnel Services, 2300 Valley Vie...",0x864e819d99a1ff99:0xeee31cc82854286c,None,32.839795,-97.020987,"(Employment agency,)",2.1,7,None,None,None,None,"[0x864e9d6ea0c9089f:0x6f90f8b0b092af49, 0x864e...",https://www.google.com/maps/place//data=!4m2!3...,"(1.16055216123327, 3.1251621244810153)",-1.696591,-2.680228,-2.006615,28.00,3590.0,3.642857,Irving,174.005102
